# DQN et Double DQN sur CartPole : du tabulaire au deep RL

**Algorithmes** : DQN et Double DQN, construits en parallele puis compares  
**Environnement** : CartPole-v1 (Gymnasium)  
**Papiers** : [Mnih et al., 2015](https://arxiv.org/abs/1312.5602) (DQN) ; [van Hasselt et al., 2016](https://arxiv.org/abs/1509.06461) (Double DQN)

| Propriete | DQN | Double DQN |
|-----------|-----|------------|
| Model-free | oui | oui |
| Value-based | oui | oui |
| Off-policy | oui | oui |
| Experience replay | oui | oui |
| Target network | oui | oui |
| Calcul du target | $\max_{a'} Q_{\text{target}}(s', a')$ | $Q_{\text{target}}(s', \arg\max_{a'} Q_{\text{online}}(s', a'))$ |

Dans le notebook precedent, on a construit un agent Q-learning tabulaire sur CartPole. L'approche repose sur une discretisation manuelle de l'espace d'observation : on decoupe chaque dimension continue en bins, puis on stocke une Q-valeur par case de la grille. Ca fonctionne sur CartPole (4 dimensions), mais le nombre de cases explose exponentiellement avec la dimension de l'espace.

DQN remplace la Q-table par un reseau de neurones. Le reseau prend l'observation brute (un vecteur continu) en entree et produit une Q-valeur par action en sortie. Deux mecanismes stabilisent l'apprentissage : un **replay buffer** et un **reseau cible**. Double DQN ajoute une correction simple qui reduit le biais de surestimation inherent au DQN.

La bonne nouvelle : DQN et Double DQN partagent **tout** sauf une seule ligne dans le calcul de la cible. On va donc construire les fondations communes d'abord, puis introduire le point de divergence, puis comparer les deux algorithmes sur le meme environnement.

## Du tabulaire au deep : pourquoi changer de representation

### Le mur du tabulaire

Le Q-learning tabulaire stocke une valeur par paire (etat, action). Quand l'espace d'etats est discret et petit (un labyrinthe 10x10 = 100 etats), ca fonctionne. Mais des qu'on a des observations continues ou de grande dimension, on se heurte a la **malediction de la dimensionnalite** :

- CartPole a 4 dimensions continues. Si on discretise chaque dimension en 20 bins, on obtient $20^4 = 160\,000$ etats. Ca passe encore.
- Un jeu Atari a des frames de $210 \times 160 \times 3$ pixels. Meme en compressant a $84 \times 84$ en niveaux de gris, le nombre d'etats possibles depasse $256^{84 \times 84} \approx 10^{17\,000}$. Aucune table ne peut stocker ca.

De plus, la Q-table n'a aucune capacite de generalisation. Deux etats voisins (position = 0.500 vs position = 0.501) sont traites comme des cases completement independantes. Si l'agent a appris que pousser a droite est bien quand la position est 0.500, il ne sait rien pour 0.501.

### La solution : approximation par reseau de neurones

On remplace la Q-table par un reseau de neurones $Q(s, a; \theta)$, parametrise par des poids $\theta$. Le reseau prend l'observation continue en entree et produit une Q-valeur pour chaque action. Des observations proches produisent naturellement des sorties proches (c'est la continuite du reseau), donc la generalisation est automatique.

Mais cette substitution cree deux problemes que le Q-learning tabulaire n'avait pas. Dans une Q-table, modifier la valeur d'une case ne change pas les autres cases. Avec un reseau, un seul gradient step modifie **tous** les poids, donc **toutes** les sorties. Deux mecanismes sont necessaires pour stabiliser l'apprentissage :

1. **Experience replay** : stocker les transitions dans un buffer et apprendre sur des mini-batches tires au hasard
2. **Target network** : utiliser une copie gelee du reseau pour calculer les cibles d'apprentissage

Ces deux mecanismes sont partages par DQN et Double DQN. La seule difference entre les deux algorithmes est dans la facon de calculer la cible TD, et on verra exactement ou ca diverge.

In [ ]:
import random
from collections import deque
from dataclasses import dataclass

import gymnasium as gym
import matplotlib.pyplot as plt
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim

from pathlib import Path
from IPython.display import Video, display
from gymnasium.wrappers import RecordVideo


plt.rcParams["figure.figsize"] = (10, 4)
plt.rcParams["figure.dpi"] = 100


## CartPole-v1 (rappel)

On reutilise le meme environnement que dans le notebook tabulaire. La difference : cette fois, on utilise les observations continues **directement**, sans discretisation.

| Dimension | Variable | Plage |
|-----------|----------|-------|
| 0 | Position du chariot | $[-4.8,\ 4.8]$ |
| 1 | Vitesse du chariot | $(-\infty,\ +\infty)$ |
| 2 | Angle du pendule | $[-0.418,\ 0.418]$ rad |
| 3 | Vitesse angulaire | $(-\infty,\ +\infty)$ |

| Action | Signification |
|--------|---------------|
| 0 | Pousser le chariot a gauche |
| 1 | Pousser le chariot a droite |

L'episode se termine si le pendule depasse 12 degres ou si le chariot sort de la zone autorisee. La recompense est $+1$ a chaque pas. Le score maximal est 500.

In [ ]:
env = gym.make("CartPole-v1")
obs, info = env.reset(seed=42)

print(f"Espace d'observation : {env.observation_space}")
print(f"  Dimension : {env.observation_space.shape[0]}")
print(f"  Bornes basses : {env.observation_space.low}")
print(f"  Bornes hautes : {env.observation_space.high}")
print(f"\nEspace d'actions : {env.action_space}")
print(f"  Nombre d'actions : {env.action_space.n}")
print(f"\nObservation initiale : {obs}")

obs, _ = env.reset(seed=42)
total_reward = 0
steps = 0
done = False
while not done:
    action = env.action_space.sample()
    obs, reward, terminated, truncated, _ = env.step(action)
    total_reward += reward
    steps += 1
    done = terminated or truncated

print(f"\nEpisode aleatoire : {steps} pas, recompense = {total_reward}")
print(f"Derniere observation : {obs}")
print(f"Type : {type(obs)}, dtype : {obs.dtype}")
env.close()

## Le reseau Q

On veut approximer la fonction de valeur d'action :

$$Q(s, a) \approx Q(s, a; \theta)$$

ou $\theta$ sont les poids du reseau. Plutot que de prendre $(s, a)$ en entree et de produire un scalaire, le reseau prend $s$ seul et produit **toutes les Q-valeurs en une passe** :

$$Q(s, \cdot\,; \theta) : \mathbb{R}^4 \to \mathbb{R}^2 \quad \text{(4 variables observées, 2 actions possibles)}$$


C'est plus efficace : pour choisir $\arg\max_a Q(s, a)$, on fait une seule forward pass au lieu de $|A|$ passes.

### Architecture
$$
\underbrace{\text{obs} \in \mathbb{R}^4}_{4 \text{ variables observées}}
\;\xrightarrow{\text{Linear}(4,64)}
\;\xrightarrow{\text{ReLU}}
\;\xrightarrow{\text{Linear}(64,64)}
\;\xrightarrow{\text{ReLU}}
\;\xrightarrow{\text{Linear}(64,2)}
\;\underbrace{\text{Q-valeurs} \in \mathbb{R}^2}_{2 \text{ actions possibles}}
$$

**Pourquoi un MLP et pas un CNN ?** L'entree est un vecteur de 4 flottants (position, vitesse, angle, vitesse angulaire). Il n'y a pas de structure spatiale a exploiter. Un CNN serait utile si l'entree etait une image (comme pour Atari, ou les pixels ont des relations de voisinage).

**Pourquoi 2 couches de 64 neurones ?** CartPole est un probleme simple. Cette capacite suffit sans risquer le surapprentissage. Sur des problemes plus complexes (Atari), on utilise des reseaux beaucoup plus grands.

**Pas d'activation en sortie.** Les Q-valeurs representent des sommes de recompenses actualisees. Elles peuvent etre negatives ou tres grandes. Une activation comme ReLU ou sigmoid ecraserait ces valeurs. La couche de sortie est lineaire.

**Comparaison avec la Q-table.** Le reseau fait exactement le meme travail que la Q-table : etant donne un etat, il renvoie une Q-valeur pour chaque action. La difference est que le reseau interpole entre les etats vus pendant l'entrainement, alors que la table ne sait rien des etats qu'elle n'a pas visites.

In [ ]:
class QNetwork(nn.Module):
    """Reseau Q(s, . ; theta) : observation -> Q-valeurs pour chaque action."""

    def __init__(self, obs_dim: int, n_actions: int, hidden_dim: int = 64):
        super().__init__()
        self.fc1 = nn.Linear(obs_dim, hidden_dim)
        self.fc2 = nn.Linear(hidden_dim, hidden_dim)
        self.fc3 = nn.Linear(hidden_dim, n_actions)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        x = F.relu(self.fc1(x))
        x = F.relu(self.fc2(x))
        return self.fc3(x)  # pas d'activation : Q-valeurs dans R


# --- Test sur une observation ---
obs_dim = 4
n_actions = 2
q_net = QNetwork(obs_dim, n_actions)

obs_tensor = torch.tensor([0.02, -0.01, 0.04, 0.02], dtype=torch.float32)
q_values = q_net(obs_tensor)
print(f"Observation : {obs_tensor.tolist()}")
print(f"Q-valeurs   : [gauche={q_values[0].item():.4f}, droite={q_values[1].item():.4f}]")
print(f"Action greedy : {'gauche' if torch.argmax(q_values).item() == 0 else 'droite'}")
print(f"\nParametres : {sum(p.numel() for p in q_net.parameters()):,}")

# Le reseau traite un batch d'observations en parallele
batch = torch.randn(32, obs_dim)
q_batch = q_net(batch)
print(f"\nBatch : {batch.shape} -> Q-values : {q_batch.shape}")

## Experience replay : pourquoi on ne peut pas apprendre en ligne

### Le probleme sans replay

En Q-learning tabulaire, mettre a jour la Q-valeur d'une case $(s, a)$ ne change rien aux autres cases. Chaque case est un nombre independant en memoire. On peut donc apprendre en ligne, transition par transition, sans probleme.

Avec un reseau de neurones, c'est fondamentalement different. **Un seul gradient step modifie tous les poids du reseau.** Si on fait une mise a jour sur la transition $(s_t, a_t, r_t, s_{t+1})$, les Q-valeurs de **tous** les autres etats changent aussi. Une experience recente unique peut ecraser ce que le reseau a appris sur des milliers de transitions precedentes.

### La correlation temporelle

Les transitions consecutives sont fortement correlees. L'agent est dans une zone de l'espace d'etats, et il y reste pendant plusieurs pas. Si on entraine le reseau sur ces transitions dans l'ordre, c'est comme entrainer un classificateur en lui montrant d'abord uniquement des chats, puis uniquement des chiens. Le gradient descend dans une direction biaisee, et le reseau oublie les experiences anterieures.

SGD (descente de gradient stochastique) suppose que les echantillons sont **i.i.d.** (independants et identiquement distribues). Des transitions consecutives violent cette hypothese : $s_{t+1}$ decoule directement de $s_t$.

### La solution : le replay buffer

On stocke chaque transition $(s, a, r, s', \text{done})$ dans un buffer circulaire de taille fixe $N$. A chaque gradient step, on tire un mini-batch de taille $B$ **uniformement au hasard** dans le buffer.

**Analogie.** Imagine un etudiant qui ne revise que son dernier cours. Il oublie tout le reste. Le replay buffer, c'est le carnet de notes : on y pioche des exercices de tous les chapitres pour reviser de facon equilibree.

Formellement, au lieu de calculer le gradient sur la transition courante :

$$\nabla_\theta \mathcal{L}(s_t, a_t)$$

on calcule le gradient sur un mini-batch echantillonne uniformement :

$$\frac{1}{B}\sum_{i \in \text{batch}} \nabla_\theta \mathcal{L}(s_i, a_i) \quad \text{avec } (s_i, a_i) \sim \text{Uniform}(D)$$

### Taille du buffer : un compromis

- Trop petit : les anciennes experiences disparaissent vite, et on retombe dans le meme probleme de correlation. Le buffer ne couvre qu'une fenetre temporelle etroite.
- Trop grand : le buffer contient beaucoup de transitions collectees par des politiques tres anciennes et mauvaises. L'agent passe du temps a apprendre de transitions obsoletes.
- En pratique : 10 000 a 1 000 000 selon la complexite de l'environnement. Pour CartPole, 10 000 suffit largement.

### Ce que le replay apporte en plus

Chaque transition peut etre reutilisee plusieurs fois. En tabulaire, une transition sert une seule mise a jour et c'est tout. Avec le replay, on extrait plus de signal de chaque interaction avec l'environnement. C'est particulierement precieux quand les interactions sont couteuses (simulation lente, robot physique).

In [ ]:
class ReplayBuffer:
    """Buffer circulaire de transitions (s, a, r, s', done)."""

    def __init__(self, capacity: int):
        self.buffer = deque(maxlen=capacity)

    def push(self, state, action, reward, next_state, done):
        self.buffer.append((state, action, reward, next_state, done))

    def sample(self, batch_size: int):
        batch = random.sample(self.buffer, batch_size)
        states, actions, rewards, next_states, dones = zip(*batch)
        return (
            torch.tensor(np.array(states), dtype=torch.float32),
            torch.tensor(actions, dtype=torch.long),
            torch.tensor(rewards, dtype=torch.float32),
            torch.tensor(np.array(next_states), dtype=torch.float32),
            torch.tensor(dones, dtype=torch.float32),
        )

    def __len__(self):
        return len(self.buffer)


buffer = ReplayBuffer(capacity=1000)
env = gym.make("CartPole-v1")
obs, _ = env.reset(seed=42)
sequential_obs = []
sequential_actions = []

for step in range(200):
    action = env.action_space.sample()
    next_obs, reward, terminated, truncated, _ = env.step(action)
    buffer.push(obs, action, reward, next_obs, terminated or truncated)
    sequential_obs.append(obs[0])
    sequential_actions.append(action)
    obs = next_obs if not (terminated or truncated) else env.reset(seed=42 + step + 1)[0]
env.close()

print(f"Taille du buffer : {len(buffer)}")
states, actions, rewards, next_states, dones = buffer.sample(32)
print(f"\nBatch de 32 transitions :")
print(f"  states      : shape {states.shape}, dtype {states.dtype}")
print(f"  actions     : shape {actions.shape}, valeurs uniques {actions.unique().tolist()}")
print(f"  rewards     : shape {rewards.shape}")
print(f"  next_states : shape {next_states.shape}")
print(f"  dones       : shape {dones.shape}, nb terminaux {int(dones.sum())}")

sampled_indices = np.random.default_rng(42).choice(len(sequential_obs), size=80, replace=False)
sampled_indices.sort()
fig, axes = plt.subplots(1, 2, figsize=(13, 4))
axes[0].plot(sequential_obs[:80], color="tab:blue", linewidth=2)
axes[0].set_title("Transitions consécutives : fortement corrélées")
axes[0].set_xlabel("Pas dans le buffer")
axes[0].set_ylabel("Position du chariot")
axes[0].grid(True, alpha=0.3)

axes[1].plot(np.asarray(sequential_obs)[sampled_indices], color="tab:orange", linewidth=2)
axes[1].set_title("Transitions échantillonnées : ordre mélangé")
axes[1].set_xlabel("Index trié après tirage aléatoire")
axes[1].set_ylabel("Position du chariot")
axes[1].grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

**Lecture.** À gauche, deux transitions voisines proviennent presque du même état physique : les présenter dans cet ordre à un réseau produit des gradients eux-mêmes corrélés. Le replay ne rend pas les données indépendantes au sens statistique, mais son tirage aléatoire casse l'ordre temporel immédiat et mélange des régions différentes de l'expérience. Le mini-batch devient ainsi plus proche de l'hypothèse i.i.d. utilisée par l'optimiseur.

## Le reseau cible : le probleme de la cible mobile

### Le probleme

La cible du Q-learning est :

$$y = r + \gamma \max_{a'} Q(s', a'; \theta)$$

Cette cible depend de $\theta$, les memes parametres qu'on est en train de modifier. Quand on fait un gradient step pour rapprocher $Q(s, a; \theta)$ de $y$, on change $\theta$, donc $y$ change aussi. On vise une cible qui se deplace a chaque tir.

En tabulaire, ce probleme n'existe quasiment pas. La mise a jour de $Q(s, a)$ ne modifie pas $Q(s', a')$ (sauf si $s' = s$, ce qui est rare). Avec un reseau, modifier $\theta$ pour un etat change les sorties pour **tous** les etats.

**Analogie.** Imagine que tu essaies de rattraper quelqu'un qui court. Si cette personne s'arrete de temps en temps pendant quelques secondes, c'est beaucoup plus facile de la rattraper. Le target network, c'est cette pause.

### La solution : deux copies du reseau

On maintient deux reseaux :
- **Le reseau en ligne** $Q(s, a; \theta)$ : mis a jour a chaque gradient step. C'est lui qui apprend.
- **Le reseau cible** $Q(s, a; \theta^-)$ : une copie gelee. Ses poids $\theta^-$ restent fixes pendant $C$ steps.

La cible devient :

$$y = r + \gamma \max_{a'} Q(s', a'; \theta^-)$$

Toutes les $C$ etapes, on synchronise : $\theta^- \leftarrow \theta$. Entre deux synchronisations, la cible est stable. Le reseau en ligne peut converger tranquillement vers une cible fixe.

### Sans target network : instabilite

Que se passe-t-il si on n'utilise pas de target network ? Les Q-valeurs oscillent violemment. Le reseau apprend a viser des cibles qui bougent en permanence. En pratique, l'entrainement diverge souvent : les Q-valeurs explosent vers l'infini, et l'agent ne converge vers aucune politique coherente. DQN ne fonctionne tout simplement pas sans ce mecanisme.

### Frequence de synchronisation $C$

- $C$ trop petit (ex: 1) : on revient au probleme initial, la cible bouge a chaque step.
- $C$ trop grand (ex: 100 000) : la cible est trop obsolete. Le reseau en ligne a beaucoup evolue, mais il est evalue par rapport a un ancien etat de lui-meme.
- En pratique : $C \in [100, 10\,000]$ selon l'environnement. Pour CartPole, $C = 100$ fonctionne bien.

In [ ]:
# Creer les deux reseaux
q_online = QNetwork(obs_dim=4, n_actions=2)
q_target = QNetwork(obs_dim=4, n_actions=2)

# Copier les poids : theta^- <- theta
q_target.load_state_dict(q_online.state_dict())

# Verifier que les sorties sont identiques au depart
obs_test = torch.randn(1, 4)
with torch.no_grad():
    print("Avant gradient step :")
    print(f"  Q en ligne : {q_online(obs_test).tolist()}")
    print(f"  Q cible    : {q_target(obs_test).tolist()}")
    print(f"  Identiques : {torch.allclose(q_online(obs_test), q_target(obs_test))}")

# Un gradient step sur q_online seulement
optimizer = optim.Adam(q_online.parameters(), lr=1e-3)
loss = q_online(obs_test).sum()
optimizer.zero_grad()
loss.backward()
optimizer.step()

# Maintenant ils divergent : c'est exactement le but
with torch.no_grad():
    print("\nApres un gradient step sur q_online :")
    print(f"  Q en ligne : {q_online(obs_test).tolist()}")
    print(f"  Q cible    : {q_target(obs_test).tolist()}")
    print(f"  Identiques : {torch.allclose(q_online(obs_test), q_target(obs_test))}")
    print("  -> Le reseau cible n'a pas bouge. C'est exactement le but.")

## Selection d'action : $\varepsilon$-greedy sur le reseau

Meme principe que dans le notebook tabulaire, mais applique aux sorties du reseau :

$$\pi(s) = \begin{cases}
\text{action aleatoire} & \text{avec probabilite } \varepsilon \\
\arg\max_a Q(s, a; \theta) & \text{avec probabilite } 1 - \varepsilon
\end{cases}$$

Deux details techniques :
1. On utilise `torch.no_grad()` pendant la selection. C'est de l'inference pure, pas d'entrainement. On evite de construire le graphe de calcul pour economiser la memoire.
2. La decroissance de $\varepsilon$ suit la meme formule exponentielle :

$$\varepsilon_{t+1} = \max(\varepsilon_{\min},\ \varepsilon_t \times d)$$

ou $d \in (0, 1)$ est le facteur de decroissance.

In [ ]:
def select_action(q_net, state, epsilon, n_actions=2):
    """Selection epsilon-greedy sur les sorties du reseau Q."""
    if random.random() < epsilon:
        return random.randrange(n_actions)
    with torch.no_grad():
        state_t = torch.tensor(state, dtype=torch.float32).unsqueeze(0)
        q_values = q_net(state_t)
        return int(torch.argmax(q_values, dim=1).item())


# --- Test ---
q_net = QNetwork(obs_dim=4, n_actions=2)
obs = np.array([0.02, -0.01, 0.04, 0.02])

print(f"Action greedy (eps=0) : {select_action(q_net, obs, epsilon=0.0)}")

# Distribution avec epsilon=0.3
counts = {0: 0, 1: 0}
for _ in range(10_000):
    a = select_action(q_net, obs, epsilon=0.3)
    counts[a] += 1
greedy_a = select_action(q_net, obs, epsilon=0.0)
print(f"\n10 000 actions avec eps=0.3 :")
print(f"  Action 0 : {counts[0]:,} ({counts[0]/100:.1f}%)")
print(f"  Action 1 : {counts[1]:,} ({counts[1]/100:.1f}%)")
print(f"  Action greedy = {greedy_a}, attendu ~85% pour celle-ci")

## La loss DQN et le role du reseau cible

On veut que le reseau produise des Q-valeurs coherentes avec l'equation de Bellman. A chaque gradient step, on mesure l'ecart entre la prediction du reseau en ligne et une cible construite a partir de la recompense et du reseau cible.

### Cible TD

Pour chaque transition $(s_i, a_i, r_i, s'_i, d_i)$ dans le batch :

$$y_i = r_i + \gamma (1 - d_i) \max_{a'} Q_{\text{target}}(s'_i, a'; \theta^-)$$

Le facteur $(1 - d_i)$ annule la contribution future si l'episode est termine ($d_i = 1$).

### Fonction de perte

$$\mathcal{L}(\theta) = \frac{1}{|B|} \sum_{i \in B} \left( y_i - Q(s_i, a_i; \theta) \right)^2$$

| Terme | Role |
|-------|------|
| $Q(s_i, a_i; \theta)$ | Prediction du reseau en ligne pour l'action effectivement prise |
| $y_i$ | Cible stable construite a partir du reseau cible |
| $\mathcal{L}(\theta)$ | Erreur quadratique moyenne entre predictions et cibles |

La cible $y_i$ est calculee avec `torch.no_grad()`. On ne propage pas le gradient a travers le reseau cible. Seuls les poids du reseau en ligne $\theta$ sont mis a jour.

**La question est : comment calculer $y_i$ exactement ?** C'est la que DQN et Double DQN divergent.

## Le point de divergence : deux façons de calculer la cible

Tout ce qui précède est partagé par DQN et Double DQN. La vraie bifurcation n'apparaît qu'au moment de calculer la cible de Bellman.

- **DQN** laisse le réseau cible faire à la fois la sélection et l'évaluation de la meilleure action future.
- **Double DQN** sépare ces deux rôles pour casser le biais de surestimation.

```mermaid
flowchart LR
classDef default fill:none,stroke:#64748b,stroke-width:1.5px
classDef primary fill:none,stroke:#2563eb,stroke-width:2px
classDef secondary fill:none,stroke:#d97706,stroke-width:2px
classDef result fill:none,stroke:#16a34a,stroke-width:2px
classDef warning fill:none,stroke:#dc2626,stroke-width:2px
S["Next state s'"]:::primary --> O["Q_online choisit argmax a*"]:::primary
S --> T["Q_target evalue la valeur future"]:::secondary
O --> U["Action retenue a*"]:::result
T --> V["Cible de bootstrap"]:::result
U --> V
```

La cellule suivante garde exactement les mêmes équations qu'avant, mais on les encapsule ensuite dans deux agents distincts pour rendre la boucle d'entraînement purement orchestratrice.

In [ ]:
def compute_dqn_loss(q_online, q_target, batch, gamma=0.99):
    """Loss DQN : le target network selectionne ET evalue."""
    states, actions, rewards, next_states, dones = batch

    q_values = q_online(states)
    q_pred = q_values.gather(1, actions.unsqueeze(1)).squeeze(1)

    with torch.no_grad():
        q_next = q_target(next_states)
        q_next_max = q_next.max(dim=1).values
        target = rewards + gamma * q_next_max * (1.0 - dones)

    loss = F.mse_loss(q_pred, target)
    td_errors = (target - q_pred).detach()
    return loss, td_errors


def compute_double_dqn_loss(q_online, q_target, batch, gamma=0.99):
    """Loss Double DQN : le reseau en ligne selectionne, le target evalue."""
    states, actions, rewards, next_states, dones = batch

    q_values = q_online(states)
    q_pred = q_values.gather(1, actions.unsqueeze(1)).squeeze(1)

    with torch.no_grad():
        best_actions = q_online(next_states).argmax(dim=1, keepdim=True)
        q_next_val = q_target(next_states).gather(1, best_actions).squeeze(1)
        target = rewards + gamma * q_next_val * (1.0 - dones)

    loss = F.mse_loss(q_pred, target)
    td_errors = (target - q_pred).detach()
    return loss, td_errors


@dataclass
class DQNConfig:
    gamma: float
    lr: float
    buffer_size: int
    batch_size: int
    target_update: int
    hidden_dim: int

class DQNAgent:
    def __init__(self, obs_dim, n_actions, config: DQNConfig):
        self.config = config
        self.n_actions = n_actions
        self.q_online = QNetwork(obs_dim, n_actions, config.hidden_dim)
        self.q_target = QNetwork(obs_dim, n_actions, config.hidden_dim)
        self.q_target.load_state_dict(self.q_online.state_dict())
        self.q_target.eval()
        self.optimizer = optim.Adam(self.q_online.parameters(), lr=config.lr)
        self.buffer = ReplayBuffer(config.buffer_size)

    def select_action(self, state, epsilon):
        return select_action(self.q_online, state, epsilon, self.n_actions)

    def store_transition(self, state, action, reward, next_state, done):
        self.buffer.push(state, action, reward, next_state, done)

    def compute_loss(self, batch):
        return compute_dqn_loss(self.q_online, self.q_target, batch, self.config.gamma)

    def learn_step(self):
        batch = self.buffer.sample(self.config.batch_size)
        self.optimizer.zero_grad()
        loss, td_errors = self.compute_loss(batch)
        loss.backward()
        self.optimizer.step()
        return loss.item(), td_errors

    def sync_target(self):
        self.q_target.load_state_dict(self.q_online.state_dict())

class DoubleDQNAgent(DQNAgent):
    def compute_loss(self, batch):
        return compute_double_dqn_loss(self.q_online, self.q_target, batch, self.config.gamma)


q_online = QNetwork(4, 2)
q_target = QNetwork(4, 2)
q_target.load_state_dict(q_online.state_dict())

demo_batch = (
    torch.randn(4, 4),
    torch.tensor([0, 1, 0, 1]),
    torch.tensor([1.0, 1.0, 1.0, 0.0]),
    torch.randn(4, 4),
    torch.tensor([0.0, 0.0, 0.0, 1.0]),
)

loss_dqn, td_dqn = compute_dqn_loss(q_online, q_target, demo_batch)
loss_ddqn, td_ddqn = compute_double_dqn_loss(q_online, q_target, demo_batch)
print(f"Loss DQN        : {loss_dqn.item():.6f}")
print(f"Loss Double DQN : {loss_ddqn.item():.6f}")
print(f"\nQuand q_online == q_target, les deux losses sont identiques :")
print(f"  Difference : {abs(loss_dqn.item() - loss_ddqn.item()):.2e}")

In [ ]:
# --- Demonstration du biais de surestimation ---
# Scenario : 10 actions, toutes avec une vraie Q-valeur de 1.0
# Deux reseaux avec du bruit d'estimation different

n_actions_demo = 10
true_q = 1.0
noise_std = 0.5
n_trials = 10_000

dqn_estimates = []
ddqn_estimates = []

rng = np.random.default_rng(42)

for _ in range(n_trials):
    # Bruit d'estimation different pour chaque reseau
    noise_target = rng.normal(0, noise_std, n_actions_demo)
    noise_online = rng.normal(0, noise_std, n_actions_demo)

    q_target_vals = true_q + noise_target
    q_online_vals = true_q + noise_online

    # DQN : max sur q_target (un seul reseau selectionne et evalue)
    dqn_est = q_target_vals.max()
    dqn_estimates.append(dqn_est)

    # Double DQN : online selectionne, target evalue
    best_a = q_online_vals.argmax()
    ddqn_est = q_target_vals[best_a]
    ddqn_estimates.append(ddqn_est)

dqn_estimates = np.array(dqn_estimates)
ddqn_estimates = np.array(ddqn_estimates)

print(f"Vraie Q-valeur : {true_q}")
print(f"Bruit std      : {noise_std}")
print(f"Nb actions     : {n_actions_demo}")
print(f"\nEstimation moyenne DQN        : {dqn_estimates.mean():.4f}  (biais = +{dqn_estimates.mean() - true_q:.4f})")
print(f"Estimation moyenne Double DQN : {ddqn_estimates.mean():.4f}  (biais = +{ddqn_estimates.mean() - true_q:.4f})")

fig, ax = plt.subplots(figsize=(10, 5))
bins = np.linspace(-0.5, 3.5, 80)
ax.hist(dqn_estimates, bins=bins, alpha=0.5, label=f"DQN (moy={dqn_estimates.mean():.3f})", color="steelblue")
ax.hist(ddqn_estimates, bins=bins, alpha=0.5, label=f"Double DQN (moy={ddqn_estimates.mean():.3f})", color="coral")
ax.axvline(true_q, color="black", linestyle="--", linewidth=2, label=f"Vraie valeur = {true_q}")
ax.set_xlabel("Estimation de la Q-valeur future")
ax.set_ylabel("Frequence")
ax.set_title(f"Biais de surestimation : DQN vs Double DQN ({n_actions_demo} actions, bruit std={noise_std})")
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
# --- Convergence des deux loss sur un batch fixe ---
# Meme buffer, memes transitions, 50 gradient steps avec chaque loss

torch.manual_seed(42)

# Creer un batch fixe
fixed_batch = (
    torch.randn(64, 4),
    torch.randint(0, 2, (64,)),
    torch.ones(64),
    torch.randn(64, 4),
    torch.zeros(64),
)

# DQN
q_online_dqn = QNetwork(4, 2)
q_target_dqn = QNetwork(4, 2)
q_target_dqn.load_state_dict(q_online_dqn.state_dict())
opt_dqn = optim.Adam(q_online_dqn.parameters(), lr=1e-3)

# Double DQN : meme initialisation
q_online_ddqn = QNetwork(4, 2)
q_target_ddqn = QNetwork(4, 2)
q_online_ddqn.load_state_dict(q_online_dqn.state_dict())
q_target_ddqn.load_state_dict(q_target_dqn.state_dict())
opt_ddqn = optim.Adam(q_online_ddqn.parameters(), lr=1e-3)

losses_dqn, losses_ddqn = [], []

for _ in range(50):
    opt_dqn.zero_grad()
    loss_d, _ = compute_dqn_loss(q_online_dqn, q_target_dqn, fixed_batch)
    loss_d.backward()
    opt_dqn.step()
    losses_dqn.append(loss_d.item())

    opt_ddqn.zero_grad()
    loss_dd, _ = compute_double_dqn_loss(q_online_ddqn, q_target_ddqn, fixed_batch)
    loss_dd.backward()
    opt_ddqn.step()
    losses_ddqn.append(loss_dd.item())

fig, ax = plt.subplots(figsize=(8, 4))
ax.plot(losses_dqn, label="DQN", color="steelblue", linewidth=2)
ax.plot(losses_ddqn, label="Double DQN", color="coral", linewidth=2, linestyle="--")
ax.set_xlabel("Gradient step")
ax.set_ylabel("Loss")
ax.set_title("Convergence sur un batch fixe (meme initialisation)")
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

print(f"Loss finale DQN        : {losses_dqn[-1]:.6f}")
print(f"Loss finale Double DQN : {losses_ddqn[-1]:.6f}")

**Lecture.** L'histogramme précédent isole le mécanisme statistique : prendre un maximum parmi plusieurs estimations bruitées pousse DQN au-dessus de la vraie valeur, tandis que la sélection et l'évaluation séparées de Double DQN ramènent la moyenne près de la cible. Le second graphique vérifie seulement que les deux objectifs sont optimisables sur le même batch fixe. Il ne faut pas lire une petite différence de loss comme un classement des agents : lorsque les réseaux online et target sont proches, leurs cibles peuvent presque coïncider.

## Les deux algorithmes complets

Les briques restent volontairement séparées : `QNetwork`, `ReplayBuffer`, calcul de loss, puis enfin `DQNAgent` ou `DoubleDQNAgent`. L'idée n'est pas de faire disparaître l'algorithme derrière une grosse classe, mais d'obtenir une boucle d'entraînement qui ne fait plus que :

1. interagir avec l'environnement ;
2. stocker la transition ;
3. demander à l'agent un `learn_step` quand le buffer est assez rempli ;
4. synchroniser périodiquement le réseau cible.

La ligne qui change vraiment entre les deux agents reste le calcul de la cible future.

## Entrainement et comparaison

### Hyperparametres

On fixe exactement les memes hyperparametres pour les deux algorithmes. La seule variable est la fonction de loss (DQN vs Double DQN). Ca permet d'isoler l'effet du decouplage selection/evaluation.

| Parametre | Valeur | Role |
|-----------|--------|------|
| `hidden_dim` | 64 | Taille des couches cachees |
| `lr` | 1e-3 | Taux d'apprentissage (Adam) |
| `gamma` | 0.99 | Facteur d'actualisation |
| `epsilon_start` | 1.0 | Exploration initiale (100% aleatoire) |
| `epsilon_decay` | 0.995 | Facteur de decroissance par episode |
| `epsilon_min` | 0.01 | Exploration minimale residuelle |
| `buffer_size` | 10 000 | Capacite du replay buffer |
| `batch_size` | 64 | Taille du mini-batch par gradient step |
| `target_update` | 100 | Frequence de synchronisation $\theta^- \leftarrow \theta$ (en steps) |
| `n_episodes` | 300 | Nombre d'episodes d'entrainement |

On entraîne en mode headless : le rendu visuel est séparé de l'entraînement pour éviter de ralentir la boucle et pour garder le notebook exécutable en environnement sans fenêtre graphique. La démonstration finale enregistrera une courte vidéo `rgb_array` de la politique greedy, affichée directement dans le notebook.

In [ ]:
# Configuration unique de la comparaison DQN / Double DQN.
SEED = 42
ENV_ID = "CartPole-v1"
RUN_DQN = True
RUN_DOUBLE_DQN = True
EVAL_SEED = 0

BUFFER_SIZE = 10_000
BATCH_SIZE = 64
GAMMA = 0.99
LR = 1e-3
EPSILON_START = 1.0
EPSILON_DECAY = 0.995
EPSILON_MIN = 0.01
TARGET_UPDATE = 100
N_EPISODES = 300
HIDDEN_DIM = 64
MAX_STEPS = 500

torch.manual_seed(SEED)
random.seed(SEED)
np.random.seed(SEED)

config = DQNConfig(
    gamma=GAMMA,
    lr=LR,
    buffer_size=BUFFER_SIZE,
    batch_size=BATCH_SIZE,
    target_update=TARGET_UPDATE,
    hidden_dim=HIDDEN_DIM,
)

In [ ]:
def train_value_agent(agent, label):
    env = gym.make(ENV_ID, render_mode="none")
    epsilon = EPSILON_START
    rewards, losses, epsilons = [], [], []
    total_steps = 0

    for episode in range(N_EPISODES):
        obs, _ = env.reset()
        episode_reward = 0.0
        episode_losses = []

        for _ in range(MAX_STEPS):
            action = agent.select_action(obs, epsilon)
            next_obs, reward, terminated, truncated, _ = env.step(action)
            done = terminated or truncated

            agent.store_transition(obs, action, reward, next_obs, done)
            episode_reward += reward
            total_steps += 1

            if len(agent.buffer) >= config.batch_size:
                loss, _ = agent.learn_step()
                episode_losses.append(loss)

            if total_steps % config.target_update == 0:
                agent.sync_target()

            obs = next_obs
            if done:
                break

        rewards.append(episode_reward)
        epsilons.append(epsilon)
        losses.append(np.mean(episode_losses) if episode_losses else 0.0)
        epsilon = max(EPSILON_MIN, epsilon * EPSILON_DECAY)

        if episode % 25 == 0 or episode == N_EPISODES - 1:
            avg = np.mean(rewards[-50:]) if len(rewards) >= 50 else np.mean(rewards)
            print(f"Episode {episode:4d} | Recompense : {episode_reward:6.1f} | Moy(50) : {avg:6.1f} | eps : {epsilon:.4f}")

    env.close()
    return rewards, losses, epsilons, total_steps

print("=" * 60)
print("ENTRAINEMENT DQN")

In [ ]:
print("=" * 60)

torch.manual_seed(SEED)
random.seed(SEED)
np.random.seed(SEED)

env_probe = gym.make(ENV_ID)
obs_dim = env_probe.observation_space.shape[0]
n_actions = env_probe.action_space.n
env_probe.close()

dqn_agent = DQNAgent(obs_dim, n_actions, config)
if RUN_DQN:
    dqn_rewards, dqn_losses, dqn_epsilons, dqn_total_steps = train_value_agent(dqn_agent, "DQN")
else:
    dqn_rewards, dqn_losses, dqn_epsilons, dqn_total_steps = [], [], [], 0
q_online_dqn_trained = dqn_agent.q_online
print(f"\nDQN termine. Steps totaux : {dqn_total_steps:,}")

In [ ]:
print("=" * 60)
print("ENTRAINEMENT DOUBLE DQN")
print("=" * 60)

torch.manual_seed(SEED)
random.seed(SEED)
np.random.seed(SEED)

ddqn_agent = DoubleDQNAgent(obs_dim, n_actions, config)
if RUN_DOUBLE_DQN:
    ddqn_rewards, ddqn_losses, ddqn_epsilons, ddqn_total_steps = train_value_agent(ddqn_agent, "Double DQN")
else:
    ddqn_rewards, ddqn_losses, ddqn_epsilons, ddqn_total_steps = [], [], [], 0
q_online_ddqn_trained = ddqn_agent.q_online
print(f"\nDouble DQN termine. Steps totaux : {ddqn_total_steps:,}")

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# --- Panel 1 : Recompenses ---
ax = axes[0]
ax.plot(dqn_rewards, alpha=0.2, color="steelblue")
ax.plot(ddqn_rewards, alpha=0.2, color="coral")
window = 50
if len(dqn_rewards) >= window:
    ma_dqn = np.convolve(dqn_rewards, np.ones(window) / window, mode="valid")
    ma_ddqn = np.convolve(ddqn_rewards, np.ones(window) / window, mode="valid")
    x = range(window - 1, window - 1 + len(ma_dqn))
    ax.plot(x, ma_dqn, color="steelblue", linewidth=2, label=f"DQN (moy. mobile {window})")
    ax.plot(x, ma_ddqn, color="coral", linewidth=2, label=f"Double DQN (moy. mobile {window})")
ax.set_xlabel("Episode")
ax.set_ylabel("Recompense")
ax.set_title("Courbe d'apprentissage")
ax.legend()
ax.grid(True, alpha=0.3)

# --- Panel 2 : Loss ---
ax = axes[1]
ax.plot(dqn_losses, alpha=0.3, color="steelblue")
ax.plot(ddqn_losses, alpha=0.3, color="coral")
if len(dqn_losses) >= 20:
    ma_ld = np.convolve(dqn_losses, np.ones(20) / 20, mode="valid")
    ma_ldd = np.convolve(ddqn_losses, np.ones(20) / 20, mode="valid")
    x20 = range(19, 19 + len(ma_ld))
    ax.plot(x20, ma_ld, color="steelblue", linewidth=2, label="DQN")
    ax.plot(x20, ma_ldd, color="coral", linewidth=2, label="Double DQN")
ax.set_xlabel("Episode")
ax.set_ylabel("Loss moyenne")
ax.set_title("Loss d'entrainement")
ax.legend()
ax.grid(True, alpha=0.3)

# --- Panel 3 : Epsilon (verification : identique pour les deux) ---
ax = axes[2]
ax.plot(dqn_epsilons, color="steelblue", linewidth=2, label="DQN")
ax.plot(ddqn_epsilons, color="coral", linewidth=2, linestyle="--", label="Double DQN")
ax.set_xlabel("Episode")
ax.set_ylabel("Epsilon")
ax.set_title("Decroissance de l'exploration (identique)")
ax.legend()
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

### Lecture des courbes

**Récompenses.** Les deux algorithmes finissent par résoudre CartPole, mais la moyenne mobile montre mieux la vitesse de stabilisation que la série brute. On lit ici une vraie progression d'apprentissage, pas simplement quelques épisodes chanceux.

**Loss.** La loss moyenne par épisode reste utile comme thermomètre, mais elle n'est pas une métrique finale de performance : un agent peut avoir une loss faible tout en gardant une politique médiocre si ses cibles sont elles-mêmes biaisées.

**Epsilon.** Les deux courbes se superposent quasiment, ce qui isole proprement l'effet du calcul de cible. Si Double DQN paraît plus stable, ce n'est pas parce qu'il explore différemment ; c'est parce qu'il sépare la sélection et l'évaluation de l'action future.

In [ ]:
def first_episode_moving_average_at_least(rewards, threshold=400, window=50):
    for end in range(window, len(rewards) + 1):
        if np.mean(rewards[end - window:end]) >= threshold:
            return end - 1
    return None


def print_metrics(name, rewards):
    last_50 = rewards[-50:]
    threshold = 400
    reached = first_episode_moving_average_at_least(rewards, threshold=threshold, window=50)
    print(f"  Moy. 50 derniers : {np.mean(last_50):6.1f}")
    print(f"  Meilleure        : {max(rewards):6.1f}")
    print(f"  Finale           : {rewards[-1]:6.1f}")
    print(f"  Episode moy. mobile >= {threshold} : {reached if reached is not None else 'non atteint'}")

print("DQN :")
print_metrics("DQN", dqn_rewards)
print(f"\nDouble DQN :")
print_metrics("DDQN", ddqn_rewards)

### Recapitulatif DQN vs Double DQN

| Dimension | DQN | Double DQN |
|-----------|-----|------------|
| Architecture du reseau | Identique | Identique |
| Replay buffer | Identique | Identique |
| Target network | Identique | Identique |
| Epsilon-greedy | Identique | Identique |
| Selection de $a^*$ dans la cible | $Q_{\text{target}}$ | $Q_{\text{online}}$ |
| Evaluation de $a^*$ dans la cible | $Q_{\text{target}}$ | $Q_{\text{target}}$ |
| Biais de surestimation | Present | Reduit |
| Stabilite de l'entrainement | Bonne | Generalement meilleure |

Meme architecture, meme replay, meme target network. La seule difference est dans le calcul du target. Sur CartPole (2 actions), l'avantage de Double DQN est subtil. Sur des environnements avec plus d'actions, la reduction du biais de surestimation fait une difference mesurable.

## Deep vs tabulaire : bilan

| Dimension | Q-learning tabulaire | DQN / Double DQN |
|-----------|---------------------|------------------|
| Representation de Q | Table discrete $|S| \times |A|$ | Reseau de neurones $Q(s,a;\theta)$ |
| Observations | Discretisees manuellement | Continues, utilisees directement |
| Generalisation | Aucune : chaque etat appris independamment | Automatique : etats proches $\to$ Q-valeurs proches |
| Memoire | $O(|S| \times |A|)$, explose avec la dimension | $O(|\theta|)$ fixe, independant de $|S|$ |
| Scalabilite | Limitee aux espaces discrets et petits | Atari, robotique, jeux 3D |
| Stabilite | Convergence garantie (sous conditions) | Moins stable, sensible aux hyperparametres |
| Efficacite en donnees | Chaque transition sert une fois | Replay : chaque transition reutilisee plusieurs fois |
| Complexite d'implementation | Faible | Moderee (replay, target network, PyTorch) |

Sur CartPole, les deux approches fonctionnent. L'avantage du DQN apparait des qu'on passe a des environnements avec des observations de grande dimension ou continues, la ou la discretisation devient impraticable.

## Demonstration des politiques apprises

L'entrainement est termine. On lance 5 episodes en mode purement greedy ($\varepsilon = 0$) pour chaque algorithme. Le reseau prend l'observation continue et choisit l'action avec la plus haute Q-valeur. Aucune exploration, aucune decision aleatoire.

In [ ]:
def evaluate_and_display_agent(q_net, label, n_episodes=5, seed=0, video_root="videos/02_dqn_cartpole"):
    """Évalue une politique greedy et affiche le dernier replay vidéo enregistré."""
    rewards = []
    safe_label = label.lower().replace(" ", "_").replace("-", "_")
    video_dir = Path(video_root) / safe_label
    video_dir.mkdir(parents=True, exist_ok=True)

    env = gym.make(ENV_ID, render_mode="rgb_array")
    env = RecordVideo(
        env,
        video_folder=str(video_dir),
        episode_trigger=lambda episode_id: episode_id == n_episodes - 1,
        name_prefix=f"{safe_label}_greedy",
    )

    q_net.eval()

    try:
        for ep in range(n_episodes):
            obs, _ = env.reset(seed=seed + ep)
            total = 0.0

            for _ in range(MAX_STEPS):
                action = select_action(q_net, obs, epsilon=0.0, n_actions=env.action_space.n)
                obs, reward, terminated, truncated, _ = env.step(action)
                total += reward
                if terminated or truncated:
                    break

            rewards.append(total)
            print(f"[{label}] Épisode {ep + 1} : {total:.0f} pas")

    finally:
        env.close()

    print(f"[{label}] Moyenne : {np.mean(rewards):.1f} pas\n")

    videos = sorted(video_dir.glob("*.mp4"), key=lambda p: p.stat().st_mtime)
    if videos:
        print(f"Replay vidéo {label} :", videos[-1])
        display(Video(str(videos[-1]), embed=True, width=480))
    else:
        print(f"Aucune vidéo générée pour {label}.")

    return rewards


print("=== DQN greedy ===")
dqn_demo = evaluate_and_display_agent(q_online_dqn_trained, "DQN")

print("=== Double DQN greedy ===")
ddqn_demo = evaluate_and_display_agent(q_online_ddqn_trained, "Double DQN")

## Conclusion et suites

DQN a transformé l'idée tabulaire du chapitre précédent en approximation de fonction : au lieu de
stocker une valeur par case discrète, un réseau apprend directement $Q(s,a)$ depuis l'observation
continue. Cette transition est le premier grand saut du projet : elle permet de traiter des états
plus riches, mais elle rend aussi l'apprentissage TD instable si l'on garde naïvement les mises à jour
du Q-learning.

Les deux stabilisateurs essentiels sont maintenant clairs. Le **replay buffer** transforme une suite
corrélée de transitions en mini-batches plus variés, ce qui rapproche l'entraînement d'une vraie
régression supervisée. Le **réseau cible** évite que la cible TD bouge à chaque gradient step. Double
DQN ajoute une correction supplémentaire : séparer l'action choisie de sa valeur évaluée pour réduire
le biais de surestimation.

La prochaine étape change de famille. Avec **REINFORCE**, on n'apprendra plus d'abord une fonction
$Q(s,a)$ pour en déduire une politique ; on optimisera directement une politique stochastique. Cela
ouvrira la porte aux méthodes policy-gradient, plus naturelles pour les actions continues et les
politiques probabilistes.